# Runner - group3-starvenus (Colab, Kaggle, local)

Drives the experiments of the generated `group3-starvenus.py` on a GPU. One notebook for all platforms; the first cells
detect where they run (**kaggle** if `KAGGLE_KERNEL_RUN_TYPE` is set or `/kaggle/input` exists, checked first because Kaggle
images ship `google.colab`; **colab** if `/var/colab/hostname` exists or `COLAB_RELEASE_TAG` / `COLAB_GPU` is set; otherwise **local**) and set the paths. Everything after that is shared. Not exported to the `.py`.

Step-by-step for Kaggle: `docs/instructions/kaggle-guide.md`.

## What differs per platform

| | Colab | Kaggle | local (tests, dry runs) |
|---|---|---|---|
| Inputs | `My Drive/csci935/` (script, requirements.txt, `Dhan-Shomadhan.zip`) | two private Datasets: `csci935-dhan-shomadhan` (the zip, or Kaggle's extracted copy) and `csci935-code` (script, requirements.txt, choices.json, optional runs.zip), found by globbing `/kaggle/input` | the repo checkout |
| Data root | `/content/Dhan-Shomadhan` | `/kaggle/tmp/Dhan-Shomadhan` (not in `/kaggle/working`, so the saved output stays small) | `Dhan-Shomadhan/` |
| Results | synced to `My Drive/csci935/_work/` after every split | copied to `/kaggle/working/out/` and `/kaggle/working/out.zip` rebuilt after every split and every batch | a temp `out/` and `out.zip` |
| Restore | runs and results from Drive | `runs.zip` and `choices.json` from `csci935-code` | none |
| GPUs | 1 (T4) | 2 (select **GPU T4 x2**): `run_parallel` runs two experiments at once, one per GPU | none |

## Colab: Drive layout (`My Drive/csci935/`)

```
csci935/
  Dhan-Shomadhan.zip        original, unmodified dataset zip (top-level folder Dhan-Shomadhan/)
  group3-starvenus.py       latest generated script  (upload again after every code change)
  requirements.txt          copy of the repo's requirements.txt
  pip_freeze.txt            written by this notebook the first time (use it to pin requirements.txt)
  _work/
    cache/  weights/  masks/   restored at the start of each session, saved after the first build
    results/choices.json       lr / input mode / backbone decisions (YOUR LOCAL COPY IS THE SOURCE OF TRUTH)
    runs/<EXP>/...             results synced back after every split (checkpoints excluded by default)
    runs/sweep/...             learning-rate sweep runs (never results)
    runs/env.json              environment of the last session
```

Handoff: download `My Drive/csci935/_work/runs/` into the local `Dhan-Shomadhan/_work/runs/` and commit.

## Order of cells (a full "Run All" executes them in this order)

1. **Settings** (first cell): `BATCHES_TO_RUN`, the only thing you edit.
2. **Setup**: helpers, platform paths, GPU check (fails early on a P100), script and requirements, pinned `torch`
   (kept if the preinstalled one matches, otherwise installed), `pip install -r requirements.txt`, one OpenCV, dataset
   copy (1106 JPEGs asserted), restore of earlier runs, `env.json`.
3. **Smoke** (`smoke` must pass) and **prepare** (`download-weights`, `prepare`; idempotent).
4. **Batches**: each batch cell runs only if its name is in `BATCHES_TO_RUN`. Cells for finished batches stay for reproducibility.
5. **Batch 4 placeholder**, then **Finish**: final sync and `out.zip`.

If a batch fails, the outputs are still synced and zipped before the error stops the run. Rerun the notebook (or the batch
cell) after fixing it: finished splits are skipped.

The gates (`gate g0`, `gate g1`), `summarize` and the tests run on the local machine, not here.

Paths in the dataset contain trailing spaces (for example `Field Background  `). The helpers pass paths as argument lists,
never through a shell string, so no quoting is needed. The cells tagged `#| lib` are plain functions with no side effects;
`tests/test_runner.py` loads exactly those cells from this notebook.

In [ ]:
#| lib
# SETTINGS and PLATFORM. Edit BATCHES_TO_RUN (and nothing else) before "Save & Run All" on Kaggle.
import os

# Which batch cells execute when the notebook runs top to bottom (cells run in notebook order, not list order).
# Method A (this session):  ["batch4"]             composites sweep + choose-composites + E3-A (splits over two GPUs)
# Optional, after batch4:   ["batch4-wf"]          E3-A-WF (white + composites, tested on field)
# Both in one session:      ["batch4", "batch4-wf"]
# Done (reproducibility): "batch1", "batch2" (E1 rows), "batch3-core", "batch3-wf", "batch3", "batch3-sweeps",
#         "run:<EXP>" (one experiment, for example "run:E3-A"), "parallel:<EXP>,<EXP>" (two at once on T4 x2).
# [] runs setup, smoke and prepare only.
BATCHES_TO_RUN = ["batch4"]

KEEP_CHECKPOINTS = False  # True also keeps best.pt (about 100-330 MB per split) in the outputs; needed only for Grad-CAM/predict
NUM_WORKERS_PARALLEL = 2  # data-loading processes per experiment when two experiments share the machine (Kaggle: 4 vCPUs)
EXPECTED_IMAGES = 1106  # JPEGs in the original dataset
GROUP_NAME = "group3-starvenus"

# Testing hooks (never needed on Colab or Kaggle). RUNNER_PLATFORM=local runs against the repo checkout.
if "RUNNER_BATCHES" in os.environ:
    BATCHES_TO_RUN = [name.strip() for name in os.environ["RUNNER_BATCHES"].split(",") if name.strip()]
SKIP_STEPS = {name.strip() for name in os.environ.get("RUNNER_SKIP", "").split(",") if name.strip()}


def detect_platform(env=None, exists=os.path.exists):
    """'kaggle', 'colab' or 'local', from the markers of the machine (never from what is importable).

    Kaggle is checked first: its images ship the google.colab package, so importability says nothing. Kaggle sets
    KAGGLE_KERNEL_RUN_TYPE and mounts /kaggle/input; Colab has /var/colab/hostname and the COLAB_RELEASE_TAG / COLAB_GPU
    variables. RUNNER_PLATFORM overrides the detection (tests, local dry runs)."""
    env = os.environ if env is None else env
    if env.get("RUNNER_PLATFORM"):
        forced = env["RUNNER_PLATFORM"].strip().lower()
        assert forced in ("colab", "kaggle", "local"), f"RUNNER_PLATFORM must be colab, kaggle or local, got {forced!r}"
        return forced
    if "KAGGLE_KERNEL_RUN_TYPE" in env or exists("/kaggle/input"):
        return "kaggle"
    if exists("/var/colab/hostname") or "COLAB_RELEASE_TAG" in env or "COLAB_GPU" in env:
        return "colab"
    return "local"


PLATFORM = detect_platform()
print("platform:", PLATFORM, "| batches:", BATCHES_TO_RUN)

## 1. Setup (run top to bottom, once per session)

The kernel never imports `torch`, `numpy` or `cv2` itself: `pip` may replace them, and everything is measured in fresh
subprocesses. So no "restart session" is needed after the installs.

In [ ]:
#| lib
# Helpers 1/5: messages, config and paths (plain functions; no side effects at import, tested in tests/test_runner.py)
import fnmatch
import json
import os
import platform as platform_module
import queue
import shutil
import signal
import subprocess
import sys
import tempfile
import threading
import time
import zipfile
from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path

DEFAULT_SPLITS = (0, 1, 2, 3, 4)
ORIGINAL_TOP_LEVEL = ("Dhan-Shomadhan_picture_Information.csv", "Field Background  ", "White Background ")


class RunnerError(RuntimeError):
    """A setup or run problem. The message is written for Nattan and says what to change."""


_LOG = {"path": None}
_LOG_LOCK = threading.Lock()


def set_log_path(path):
    """Mirror everything printed by `say` into this file (it ends up in out.zip on Kaggle)."""
    _LOG["path"] = None if path is None else Path(path)
    if _LOG["path"] is not None:
        _LOG["path"].parent.mkdir(parents=True, exist_ok=True)


def say(text=""):
    """print + append to the runner log."""
    print(text, flush=True)
    path = _LOG["path"]
    if path is not None:
        with _LOG_LOCK, open(path, "a", encoding="utf-8") as handle:
            handle.write(str(text) + "\n")


def kill_process_group(proc):
    """Kill ``proc`` and every process left in its group (`stream` starts commands with start_new_session=True, so the
    group is the command's own: its worker processes and nothing else), then reap ``proc``. Does nothing for a group
    that is already gone, and falls back to ``proc.kill()`` where process groups do not exist (Windows)."""
    pid = getattr(proc, "pid", None)
    if not isinstance(pid, int):
        return  # a test double
    if hasattr(os, "killpg"):
        try:
            os.killpg(pid, signal.SIGKILL)
        except (ProcessLookupError, PermissionError):
            pass
    elif proc.poll() is None:
        proc.kill()
    try:
        proc.wait(timeout=10)
    except subprocess.TimeoutExpired:
        pass


def warn(text):
    """A loud multi-line warning (visible when scrolling a long Kaggle log)."""
    bar = "!" * 78
    say(f"\n{bar}\nWARNING: {text}\n{bar}\n")


@dataclass
class Config:
    platform: str  # colab | kaggle | local
    group_name: str
    script_name: str
    script_path: Path  # where the generated .py runs from
    requirements_path: Path
    data_root: Path  # dataset copy; _work/ (cache, splits, runs, weights) lives inside it
    scratch: Path  # parent area for staging, never persisted
    persist: str  # "drive" (Colab) or "outdir" (Kaggle, local): where finished runs are copied to
    device: str  # value of the CLI's --device flag
    require_gpu: bool
    drive: Path | None = None  # Colab: My Drive/csci935
    out_dir: Path | None = None  # Kaggle: /kaggle/working/out
    out_zip: Path | None = None  # Kaggle: /kaggle/working/out.zip
    input_root: Path | None = None  # Kaggle: /kaggle/input
    repo_root: Path | None = None  # local: the repository checkout

    @property
    def work(self) -> Path:
        return self.data_root / "_work"

    @property
    def runs_dir(self) -> Path:
        return self.work / "runs"

    @property
    def choices_path(self) -> Path:
        return self.work / "results" / "choices.json"

    @property
    def drive_work(self) -> Path:
        return self.drive / "_work"

    @property
    def drive_runs(self) -> Path:
        return self.drive / "_work" / "runs"

    @property
    def env_json_path(self) -> Path:
        return (self.drive_runs if self.persist == "drive" else self.out_dir) / "env.json"

    @property
    def log_path(self) -> Path:
        return (self.out_dir if self.persist == "outdir" else self.scratch) / "runner_log.txt"


def free_gb(path) -> float:
    return shutil.disk_usage(path).free / 1e9


def pick_scratch_dir(candidates, min_free_gb=8.0, say=say) -> Path:
    """The first candidate directory that can be created, is writable and has `min_free_gb` free.

    On Kaggle the data root must not live in /kaggle/working (everything there is saved as notebook output)."""
    problems = []
    for candidate in candidates:
        candidate = Path(candidate)
        try:
            candidate.mkdir(parents=True, exist_ok=True)
            probe = candidate / ".write_test"
            probe.write_text("ok")
            probe.unlink()
            free = free_gb(candidate)
        except OSError as exc:
            problems.append(f"{candidate}: not writable ({exc.__class__.__name__})")
            continue
        if free < min_free_gb:
            problems.append(f"{candidate}: only {free:.1f} GB free")
            continue
        say(f"scratch area: {candidate} ({free:.1f} GB free)")
        return candidate
    raise RunnerError("no writable scratch area with enough space for the dataset copy:\n  " + "\n  ".join(problems))


def resolve_config(platform, group_name, env=None, kaggle_root="/kaggle", colab_root="/content", cwd=None,
                   min_free_gb=8.0, say=say) -> Config:
    """All paths of one session. Everything after this is shared by the three platforms."""
    env = dict(os.environ) if env is None else env
    script_name = f"{group_name}.py"
    if platform == "colab":
        content = Path(colab_root)
        return Config(
            platform="colab", group_name=group_name, script_name=script_name,
            script_path=content / script_name, requirements_path=content / "requirements.txt",
            data_root=content / "Dhan-Shomadhan", scratch=content, persist="drive", device="cuda", require_gpu=True,
            drive=content / "drive" / "MyDrive" / "csci935",
        )
    if platform == "kaggle":
        root = Path(kaggle_root)
        scratch = pick_scratch_dir([root / "tmp", root / "temp", Path(tempfile.gettempdir())], min_free_gb, say)
        code = scratch / "code"
        return Config(
            platform="kaggle", group_name=group_name, script_name=script_name,
            script_path=code / script_name, requirements_path=code / "requirements.txt",
            data_root=scratch / "Dhan-Shomadhan", scratch=scratch, persist="outdir", device="cuda", require_gpu=True,
            out_dir=root / "working" / "out", out_zip=root / "working" / "out.zip", input_root=root / "input",
        )
    if platform == "local":
        repo = Path(env.get("RUNNER_REPO") or cwd or Path.cwd()).resolve()
        base = Path(env.get("RUNNER_OUT_BASE") or tempfile.mkdtemp(prefix="runner_out_"))
        return Config(
            platform="local", group_name=group_name, script_name=script_name,
            script_path=repo / script_name, requirements_path=repo / "requirements.txt",
            data_root=Path(env.get("RUNNER_DATA_ROOT") or repo / "Dhan-Shomadhan"), scratch=base,
            persist="outdir", device="auto", require_gpu=False,
            out_dir=base / "out", out_zip=base / "out.zip", repo_root=repo,
        )
    raise RunnerError(f"unknown platform {platform!r}")


def check_python(version_info=None):
    """The generated script uses Python 3.12 APIs (pathlib.Path.walk)."""
    version_info = sys.version_info if version_info is None else version_info
    if tuple(version_info[:2]) < (3, 12):
        raise RunnerError(
            f"Python {version_info[0]}.{version_info[1]} is too old: group3-starvenus.py needs 3.12 or newer "
            "(pathlib.Path.walk). On Kaggle use the latest environment: Settings > Environment > "
            "'Always use latest environment'.")


def ensure_internet(urlopen=None, url="https://pypi.org/simple/pip/", timeout=15):
    """Fail early (before any long step) when the session has no internet."""
    if urlopen is None:
        import urllib.request

        urlopen = urllib.request.urlopen
    try:
        urlopen(url, timeout=timeout).close()
    except Exception as exc:  # noqa: BLE001 - any network failure means the same thing here
        raise RunnerError(
            f"no internet ({exc.__class__.__name__}: {exc}). pip install and download-weights need it. "
            "Kaggle: Session options (right sidebar) > Internet > On. The switch needs a phone-verified account "
            "(Settings > Phone verification).") from exc

In [ ]:
#| lib
# Helpers 2/5: GPU checks, torch/requirements installation, environment probe
# The notebook kernel never imports torch, numpy or cv2: pip may replace them, and a kernel that imported the old
# build would report the wrong versions. Everything is probed in a fresh subprocess instead.
GPU_QUERY = ["nvidia-smi", "--query-gpu=index,name,compute_cap,driver_version,memory.total", "--format=csv,noheader,nounits"]
MIN_COMPUTE_CAPABILITY = (7, 5)  # the pinned torch 2.11 builds (CUDA 13.0 and 12.8) ship kernels for Turing (sm_75) and newer
TORCH_INDEX = "https://download.pytorch.org/whl"


def parse_gpus(text):
    """Rows of `nvidia-smi --query-gpu=index,name,compute_cap,driver_version,memory.total --format=csv,noheader,nounits`."""
    gpus = []
    for line in text.strip().splitlines():
        parts = [part.strip() for part in line.split(",")]
        if len(parts) < 5:
            continue
        try:
            major, minor = (int(x) for x in parts[2].split("."))
        except ValueError:
            major, minor = 0, 0
        gpus.append({"index": int(parts[0]), "name": parts[1], "capability": (major, minor), "driver": parts[3],
                     "memory_mb": int(float(parts[4])) if parts[4].replace(".", "").isdigit() else None})
    return gpus


def query_gpus(run=subprocess.run):
    try:
        result = run(GPU_QUERY, capture_output=True, text=True, timeout=60)
    except (OSError, subprocess.SubprocessError):
        return []
    return parse_gpus(result.stdout) if result.returncode == 0 else []


def check_gpus(gpus, platform, say=say):
    """Print the GPUs and fail early on no GPU or an unsupported one (P100 on Kaggle). Returns the GPU names."""
    where = {
        "kaggle": "Kaggle: Session options (right sidebar) > Accelerator > 'GPU T4 x2', then Run All again.",
        "colab": "Colab: Runtime > Change runtime type > T4 GPU, then restart.",
    }.get(platform, "")
    if not gpus:
        raise RunnerError("no GPU found (nvidia-smi lists none). " + where)
    for gpu in gpus:
        say(f"GPU {gpu['index']}: {gpu['name']} (compute capability {gpu['capability'][0]}.{gpu['capability'][1]}, "
            f"{gpu['memory_mb']} MB, driver {gpu['driver']})")
    for gpu in gpus:
        if gpu["capability"] < MIN_COMPUTE_CAPABILITY or "P100" in gpu["name"]:
            raise RunnerError(
                f"{gpu['name']} (compute capability {gpu['capability'][0]}.{gpu['capability'][1]}) cannot run the pinned "
                "torch 2.11: its CUDA 13.0 and 12.8 builds contain no kernels for Pascal (sm_60); only a CUDA 12.6 "
                "build does, and a different torch/CUDA build and GPU type than the T4 results would break the "
                "'all splits of one experiment on the same GPU type' rule. " + where)
    names = [gpu["name"] for gpu in gpus]
    if len(set(names)) > 1:
        warn(f"the GPUs are different models ({', '.join(names)}); run_parallel puts one experiment on each, so the "
             "two experiments would use different hardware.")
    if not any("T4" in name for name in names):
        warn(f"GPU is {names[0]!r}, not a T4. Every committed result ran on a Tesla T4; mixing GPU types inside one "
             "experiment changes timings and can change numbers slightly. Prefer T4.")
    if platform == "kaggle" and len(gpus) == 1:
        say("note: only one GPU is visible. Select 'GPU T4 x2' to use run_parallel; with one GPU it runs sequentially.")
    return names


# ---- environment probe (fresh subprocess) -------------------------------------------------------------------

PROBE_CODE = r'''
import json, platform
out = {"python": platform.python_version()}
try:
    import torch
    out["torch"] = torch.__version__
    out["cuda"] = torch.version.cuda
    out["cuda_available"] = bool(torch.cuda.is_available())
    out["arch_list"] = list(torch.cuda.get_arch_list()) if out["cuda_available"] else []
    devices = []
    for i in range(torch.cuda.device_count()):
        entry = {"index": i, "name": torch.cuda.get_device_name(i), "capability": list(torch.cuda.get_device_capability(i))}
        try:
            entry["op_ok"] = bool((torch.ones(8, device=f"cuda:{i}") + 1).sum().item() == 16.0)
        except Exception as exc:
            entry["op_ok"] = False
            entry["error"] = f"{type(exc).__name__}: {exc}"[:300]
        devices.append(entry)
    out["devices"] = devices
except Exception as exc:
    out["torch_error"] = f"{type(exc).__name__}: {exc}"[:500]
for name, module in (("torchvision", "torchvision"), ("opencv", "cv2"), ("numpy", "numpy")):
    try:
        out[name] = __import__(module).__version__
    except Exception as exc:
        out[name + "_error"] = f"{type(exc).__name__}: {exc}"[:300]
print("PROBE_JSON " + json.dumps(out))
'''


def probe_environment(python=None, run=subprocess.run):
    """Versions and a real CUDA operation per GPU, measured in a fresh interpreter. Never raises."""
    python = sys.executable if python is None else python
    try:
        result = run([str(python), "-c", PROBE_CODE], capture_output=True, text=True, timeout=600)
    except (OSError, subprocess.SubprocessError) as exc:
        return {"probe_error": str(exc)}
    for line in result.stdout.splitlines():
        if line.startswith("PROBE_JSON "):
            return json.loads(line[len("PROBE_JSON "):])
    return {"probe_error": (result.stderr or result.stdout or "no output")[-500:]}


def base_version(version):
    return str(version or "").split("+")[0]


def probe_cuda_ok(info, expected_devices=1):
    devices = info.get("devices") or []
    return bool(info.get("cuda_available")) and len(devices) >= expected_devices and all(d.get("op_ok") for d in devices)


def parse_requirements(text):
    """{'torch': '2.11.0', ...} for the pinned `name==version` lines (comments and options are ignored)."""
    pins = {}
    for line in text.splitlines():
        line = line.split("#")[0].strip()
        if "==" in line and not line.startswith("-"):
            name, version = line.split("==", 1)
            pins[name.strip().lower()] = version.strip()
    return pins


def filter_requirements(text, exclude):
    """requirements.txt without the pins of `exclude` (used when the torch pin cannot be honoured)."""
    kept = []
    for line in text.splitlines():
        name = line.split("#")[0].split("==")[0].strip().lower()
        if name not in exclude:
            kept.append(line)
    return "\n".join(kept) + "\n"


def driver_major(driver):
    try:
        return int(str(driver).split(".")[0])
    except ValueError:
        return 0


def torch_candidates(driver, pins, python=None):
    """pip commands to try, best first, for the pinned torch/torchvision.

    The Colab results were produced with torch 2.11.0+cu130 (the PyPI default for 2.11), which needs a driver of 580 or
    newer. With an older driver (Kaggle has used 5xx) the CUDA 12.8 build of the same release is used; both contain
    kernels for the T4 (sm_75). The `==2.11.0+cuXXX` local version still satisfies `torch==2.11.0` in requirements.txt."""
    python = sys.executable if python is None else python
    torch_v, tv_v = pins["torch"], pins["torchvision"]
    base = [str(python), "-m", "pip", "install", "-q", "--disable-pip-version-check"]
    pypi = {"label": "PyPI default (CUDA 13.0)", "cmd": [*base, f"torch=={torch_v}", f"torchvision=={tv_v}"]}

    def indexed(tag):
        return {"label": f"{tag} wheels from download.pytorch.org",
                "cmd": [*base, f"torch=={torch_v}+{tag}", f"torchvision=={tv_v}+{tag}", "--extra-index-url", f"{TORCH_INDEX}/{tag}"]}

    if driver_major(driver) >= 580:
        return [pypi, indexed("cu128")]
    return [indexed("cu128"), indexed("cu126")]


def ensure_torch(stream, pins, driver, expected_devices, probe=probe_environment, python=None, say=say):
    """Make the pinned torch/torchvision importable with working CUDA, with the least download.

    1. The installed torch already has the pinned version and a CUDA operation works on every GPU: keep it.
    2. Otherwise install the pin (see torch_candidates) and probe again, first working candidate wins.
    3. If no candidate works but the preinstalled torch does, continue with it and print a loud warning.
    Returns {'honoured': bool, 'how': str, 'info': probe dict}; raises RunnerError when no torch can use the GPU."""
    info = probe()
    if (base_version(info.get("torch")) == pins["torch"] and base_version(info.get("torchvision")) == pins["torchvision"]
            and probe_cuda_ok(info, expected_devices)):
        say(f"torch {info['torch']} / torchvision {info['torchvision']} already match the pin and CUDA works: nothing to install")
        return {"honoured": True, "how": "preinstalled", "info": info}
    say(f"installed torch: {info.get('torch') or info.get('torch_error') or info.get('probe_error')}; "
        f"pin is torch=={pins['torch']}, torchvision=={pins['torchvision']}, driver {driver}")
    for candidate in torch_candidates(driver, pins, python):
        say(f"installing the pinned torch: {candidate['label']}")
        try:
            stream(candidate["cmd"])
        except RunnerError as exc:
            say(f"  failed: {exc}")
            continue
        info = probe()
        if (base_version(info.get("torch")) == pins["torch"] and probe_cuda_ok(info, expected_devices)):
            return {"honoured": True, "how": candidate["label"], "info": info}
        say(f"  installed, but CUDA does not work with it: {info.get('torch_error') or info.get('devices')}")
    if probe_cuda_ok(info, expected_devices):
        warn(f"THE TORCH PIN COULD NOT BE HONOURED. Continuing with torch {info.get('torch')} / torchvision "
             f"{info.get('torchvision')} (pin: {pins['torch']} / {pins['torchvision']}). Numbers may differ slightly from "
             "the Colab T4 results. The actual versions are recorded in env.json.")
        return {"honoured": False, "how": "kept after failed install", "info": info}
    raise RunnerError(f"no torch build can use the GPU: {info.get('torch_error') or info.get('devices') or info.get('probe_error')}")


OPENCV_PACKAGES = ("opencv-python", "opencv-contrib-python", "opencv-python-headless", "opencv-contrib-python-headless")


def install_requirements(stream, requirements_path, torch_honoured, scratch, probe=probe_environment, python=None, say=say):
    """Uninstall every OpenCV distribution (they all write the same cv2/ folder), install the pinned requirements,
    assert OpenCV 4.12. When the torch pin was not honoured, torch/torchvision are left out so pip keeps them."""
    python = sys.executable if python is None else python
    stream([python, "-m", "pip", "uninstall", "-y", "-q", *OPENCV_PACKAGES])
    requirements = Path(requirements_path)
    if not torch_honoured:
        filtered = Path(scratch) / "requirements.filtered.txt"
        filtered.write_text(filter_requirements(requirements.read_text(), {"torch", "torchvision"}))
        requirements = filtered
    stream([python, "-m", "pip", "install", "-q", "--disable-pip-version-check", "-r", requirements])
    info = probe()
    version = info.get("opencv", "")
    if not str(version).startswith("4.12."):
        raise RunnerError(f"expected OpenCV 4.12.x after installing the requirements, got {version or info.get('opencv_error')}")
    say(f"OpenCV {version}")
    return info


def pip_freeze(python=None, run=subprocess.run):
    python = sys.executable if python is None else python
    return run([str(python), "-m", "pip", "freeze"], capture_output=True, text=True, check=True).stdout


def build_env_record(cfg, gpus, info, torch_result, layout=None, now=None):
    """The environment.json written next to the runs. Keeps the keys of the earlier Colab env.json."""
    now = datetime.now(timezone.utc) if now is None else now
    record = {
        "recorded_at": now.isoformat(),
        "platform": cfg.platform,
        "gpu": gpus[0]["name"] if gpus else None,
        "gpus": [gpu["name"] for gpu in gpus],
        "driver": gpus[0]["driver"] if gpus else None,
        "python": info.get("python") or platform_module.python_version(),
        "torch": info.get("torch"),
        "torchvision": info.get("torchvision"),
        "opencv": info.get("opencv"),
        "numpy": info.get("numpy"),
        "cuda": info.get("cuda"),
        "torch_pin_honoured": None if torch_result is None else torch_result["honoured"],
        "torch_install": None if torch_result is None else torch_result["how"],
    }
    if layout:
        record["dataset_layout"] = layout
    return record

In [ ]:
#| lib
# Helpers 3/5: Kaggle inputs, dataset copy, restore of earlier runs
JPEG_SUFFIXES = (".jpg", ".jpeg")
JUNK_NAMES = {".DS_Store", "__MACOSX"}


def is_jpeg(name):
    return Path(name).suffix.lower() in JPEG_SUFFIXES and not Path(name).name.startswith("._")


def walk_limited(root, max_depth):
    """os.walk that does not descend below max_depth levels under root."""
    root = Path(root)
    for dirpath, dirnames, filenames in os.walk(root):
        depth = len(Path(dirpath).relative_to(root).parts)
        if depth >= max_depth:
            dirnames[:] = []
        yield Path(dirpath), dirnames, filenames


def count_images(data_root):
    """Dataset JPEGs under data_root, excluding generated data in _work/."""
    data_root = Path(data_root)
    if not data_root.is_dir():
        return 0
    total = 0
    for dirpath, dirnames, filenames in os.walk(data_root):
        if Path(dirpath) == data_root and "_work" in dirnames:
            dirnames.remove("_work")
        total += sum(1 for name in filenames if is_jpeg(name))
    return total


def find_jpegs(root, exclude=None):
    found = []
    for dirpath, dirnames, filenames in os.walk(root):
        if exclude is not None and Path(dirpath) == Path(exclude):
            dirnames[:] = []
            continue
        found.extend(Path(dirpath) / name for name in filenames if is_jpeg(name))
    return found


def dataset_top(jpegs):
    """The deepest folder that holds every image: the dataset root, whatever Kaggle named the folders."""
    return Path(os.path.commonpath([str(p.parent) for p in jpegs]))


def top_level_names(path):
    """repr() of the entries directly under path, so that trailing spaces are visible."""
    return sorted(repr(p.name) for p in Path(path).iterdir() if p.name not in ("_work", *JUNK_NAMES))


def describe_tree(root, max_entries=60, max_depth=3):
    lines = []
    root = Path(root)
    if not root.exists():
        return f"(missing: {root})"
    for dirpath, dirnames, filenames in walk_limited(root, max_depth):
        dirnames.sort()
        rel = dirpath.relative_to(root)
        indent = "  " * len(rel.parts)
        lines.append(f"{indent}{repr(dirpath.name) if rel.parts else str(root)}/  ({len(filenames)} files)")
        for name in sorted(filenames)[:5]:
            lines.append(f"{indent}  {name!r}")
        if len(lines) >= max_entries:
            lines.append("  ...")
            break
    return "\n".join(lines)


# /kaggle/input/datasets/<owner>/<slug>/<file> is depth 4; allow two more levels for version folders.
KAGGLE_SEARCH_DEPTH = 6


def discover_kaggle_inputs(input_root, script_name, expected_images=1106):
    """Find the code dataset and the dataset copy under /kaggle/input by content, not by slug.

    Returns {'code_dir', 'script', 'requirements', 'choices', 'runs_zip', 'runs_dir', 'data_kind', 'data_source'};
    'data_kind' is 'zip' (the original Dhan-Shomadhan.zip) or 'tree' (Kaggle extracted it). Raises RunnerError with a
    listing of what is mounted when something is missing."""
    input_root = Path(input_root)
    if not input_root.is_dir():
        raise RunnerError(f"{input_root} does not exist: this is not a Kaggle session, or no dataset is attached.")
    listing = "\n" + describe_tree(input_root)

    hits = [Path(dp) / name for dp, _, files in walk_limited(input_root, KAGGLE_SEARCH_DEPTH) for name in files if name == script_name]
    code_dirs = sorted({hit.parent for hit in hits})
    if not code_dirs:
        raise RunnerError(f"{script_name} not found under {input_root}. Attach the dataset 'csci935-code' (right sidebar > "
                          f"Add Input) and make sure it contains {script_name}. What is mounted:{listing}")
    if len(code_dirs) > 1:
        raise RunnerError(f"{script_name} exists in several attached datasets: {[str(d) for d in code_dirs]}. Attach only one.")
    code_dir = code_dirs[0]
    requirements = code_dir / "requirements.txt"
    if not requirements.is_file():
        raise RunnerError(f"{code_dir} has {script_name} but no requirements.txt. What is there:\n" + describe_tree(code_dir))

    zips = [Path(dp) / name for dp, _, files in walk_limited(input_root, KAGGLE_SEARCH_DEPTH) for name in files
            if name.lower().endswith(".zip") and Path(dp) != code_dir and "dhan" in name.lower()]
    if len(zips) > 1:
        raise RunnerError(f"several dataset zips are attached: {[str(z) for z in zips]}. Attach only one.")
    if zips:
        data_kind, data_source = "zip", zips[0]
    else:
        jpegs = find_jpegs(input_root, exclude=code_dir)
        if not jpegs:
            raise RunnerError("the dataset images were not found: expected Dhan-Shomadhan.zip or the extracted JPEGs in an "
                              f"attached dataset (suggested name 'csci935-dhan-shomadhan'). What is mounted:{listing}")
        data_kind, data_source = "tree", dataset_top(jpegs)
        if len(jpegs) != expected_images:
            raise RunnerError(f"found {len(jpegs)} JPEGs under {data_source}, expected {expected_images}. What is mounted:{listing}")
    runs_zip = code_dir / "runs.zip"
    runs_dir = code_dir / "runs"
    choices = code_dir / "choices.json"
    return {
        "code_dir": code_dir, "script": code_dir / script_name, "requirements": requirements,
        "choices": choices if choices.is_file() else None,
        "runs_zip": runs_zip if runs_zip.is_file() else None,
        "runs_dir": runs_dir if runs_dir.is_dir() else None,
        "data_kind": data_kind, "data_source": data_source,
    }


def copy_tree_plain(src, dst, move=False):
    """Copy (or move) a tree without copying permission bits, so a read-only source (/kaggle/input) gives a writable copy."""
    src, dst = Path(src), Path(dst)
    for dirpath, dirnames, filenames in os.walk(src):
        dirnames[:] = [d for d in dirnames if d != "_work" or Path(dirpath) != src]
        target_dir = dst / Path(dirpath).relative_to(src)
        target_dir.mkdir(parents=True, exist_ok=True)
        for name in filenames:
            if name in JUNK_NAMES:
                continue
            if move:
                shutil.move(str(Path(dirpath) / name), str(target_dir / name))
            else:
                shutil.copyfile(Path(dirpath) / name, target_dir / name)


def materialise_dataset(kind, source, data_root, scratch, expected_images=1106, say=say):
    """Put a complete, writable copy of the original dataset at data_root; returns a layout record.

    'zip': extract Dhan-Shomadhan.zip into scratch, then move its dataset folder to data_root.
    'tree': copy the (possibly Kaggle-extracted, possibly renamed) tree into data_root.
    Either way the dataset folder is found as the deepest folder that contains all JPEGs, so no folder name matters.
    Decides on completeness, not existence: an interrupted copy is repaired, and _work/ is never touched."""
    data_root = Path(data_root)
    have = count_images(data_root)
    if have == expected_images:
        say(f"dataset already complete at {data_root} ({have} images)")
        return {"kind": "existing", "data_root_top_level": top_level_names(data_root)}
    layout = {"kind": kind, "source": str(source)}
    if kind == "zip":
        stage = Path(scratch) / "_dataset_stage"
        shutil.rmtree(stage, ignore_errors=True)
        stage.mkdir(parents=True)
        size = Path(source).stat().st_size / 1e9
        say(f"extracting {source} ({size:.2f} GB) to {stage} ...")
        with zipfile.ZipFile(source) as archive:
            names = [n for n in archive.namelist() if not any(part in JUNK_NAMES for part in Path(n).parts)]
            archive.extractall(stage, members=names)
        top = dataset_top(find_jpegs(stage))
        layout["zip_top_level_entries"] = sorted({repr(Path(n).parts[0]) for n in names if Path(n).parts})
        move = True
    else:
        top, move = Path(source), False
    layout["source_top_level"] = top_level_names(top)
    say(f"top-level entries of the dataset folder (repr shows trailing spaces): {layout['source_top_level']}")
    copy_tree_plain(top, data_root, move=move)
    if kind == "zip":
        shutil.rmtree(Path(scratch) / "_dataset_stage", ignore_errors=True)
    found = count_images(data_root)
    if found != expected_images:
        raise RunnerError(f"expected {expected_images} images in the original dataset, found {found} under {data_root}\n"
                          + describe_tree(data_root))
    layout["data_root_top_level"] = top_level_names(data_root)
    changed = sorted(set(layout["data_root_top_level"]) - {repr(n) for n in ORIGINAL_TOP_LEVEL})
    layout["differs_from_original_names"] = changed
    if changed:
        say(f"note: folder names differ from the original download (the code never depends on them): {changed}")
    say(f"dataset OK: {found} images under {data_root}")
    return layout


def map_member(parts):
    """Where a file of runs.zip (or of an extracted runs/ folder) goes, relative to _work/; None to skip.

    Accepts the layout of out.zip ('runs/<EXP>/...', 'results/choices.json'), of a copied '_work/' tree, and bare
    '<EXP>/split_k/...' folders (which belong to runs/)."""
    parts = [p for p in parts if p not in ("", ".")]
    if not parts or ".." in parts or any(p in JUNK_NAMES for p in parts):
        return None
    if parts[0] == "_work":
        parts = parts[1:]
    if not parts:
        return None
    if parts[0] in ("runs", "results"):
        return Path(*parts)
    if len(parts) == 1:  # loose top-level file such as env.json, pip_freeze.txt, runner_log.txt: not a run
        return None
    return Path("runs", *parts)


def unpack_runs_zip(zip_path, work, say=say):
    """Extract an earlier out.zip / runs.zip into _work/; returns the number of files written."""
    work, written = Path(work), 0
    with zipfile.ZipFile(zip_path) as archive:
        for info in archive.infolist():
            if info.is_dir():
                continue
            target = map_member(Path(info.filename).parts)
            if target is None:
                continue
            destination = work / target
            destination.parent.mkdir(parents=True, exist_ok=True)
            with archive.open(info) as source, open(destination, "wb") as handle:
                shutil.copyfileobj(source, handle)
            written += 1
    say(f"restored {written} files from {zip_path} into {work}")
    return written


def restore_from_code_dataset(inputs, work, say=say):
    """Kaggle 'restore': runs.zip (or an extracted runs/ folder) and choices.json from csci935-code into _work/,
    so that finished splits are skipped. choices.json is applied last: your local copy is the source of truth."""
    work = Path(work)
    work.mkdir(parents=True, exist_ok=True)
    restored = 0
    if inputs.get("runs_zip"):
        restored += unpack_runs_zip(inputs["runs_zip"], work, say)
    elif inputs.get("runs_dir"):
        for dirpath, _, filenames in os.walk(inputs["runs_dir"]):
            for name in filenames:
                rel = (Path(dirpath) / name).relative_to(inputs["runs_dir"])
                target = map_member(("runs", *rel.parts))
                if target is not None:
                    (work / target).parent.mkdir(parents=True, exist_ok=True)
                    shutil.copyfile(Path(dirpath) / name, work / target)
                    restored += 1
        say(f"restored {restored} files from the extracted {inputs['runs_dir']}")
    else:
        say("no runs.zip in the code dataset: starting without earlier runs")
    if inputs.get("choices"):
        destination = work / "results" / "choices.json"
        destination.parent.mkdir(parents=True, exist_ok=True)
        shutil.copyfile(inputs["choices"], destination)
        say(f"choices.json from the code dataset -> {destination}")
    else:
        say("no choices.json in the code dataset (the sweeps will create the learning rates)")
    return restored


def finished_splits(runs_dir):
    """{'E0-rect': [0, 1, 2, 3, 4], ...}: splits with a metrics.json, sweeps and scratch excluded."""
    result = {}
    runs_dir = Path(runs_dir)
    if not runs_dir.is_dir():
        return result
    for exp_dir in sorted(p for p in runs_dir.iterdir() if p.is_dir() and p.name not in ("sweep", "scratch")):
        splits = sorted(int(p.name.split("_")[1]) for p in exp_dir.glob("split_*")
                        if (p / "metrics.json").is_file() and p.name.split("_")[1].isdigit())
        if splits:
            result[exp_dir.name] = splits
    return result

In [ ]:
#| lib
# Helpers 4/5: syncing results (Drive on Colab, out/ + out.zip on Kaggle) and the hardware rule
CHECKPOINT_PATTERNS = ("*.pt", "*.pth")
TEMP_PATTERNS = ("*.tmp*",)  # atomic-write temp files of a run that is still writing


def _wanted(name, patterns):
    return not any(fnmatch.fnmatch(name, pattern) for pattern in patterns)


def _unchanged(src, dst):
    try:
        a, b = src.stat(), dst.stat()
    except OSError:
        return False
    return a.st_size == b.st_size and abs(a.st_mtime - b.st_mtime) < 2


def sync_tree(src, dst, keep_checkpoints=True, skip=(), say=say, quiet=False):
    """Merge src into dst (overwrite changed files only). Skips *.pt / *.pth unless keep_checkpoints, any `skip` pattern,
    and temp files of in-flight writes. Returns the number of files copied."""
    src, dst = Path(src), Path(dst)
    patterns = (*skip, *TEMP_PATTERNS, *([] if keep_checkpoints else CHECKPOINT_PATTERNS))
    copied = 0
    for dirpath, dirnames, filenames in os.walk(src):
        dirnames[:] = [d for d in dirnames if _wanted(d, skip)]
        target_dir = dst / Path(dirpath).relative_to(src)
        for name in filenames:
            if not _wanted(name, patterns):
                continue
            source, target = Path(dirpath) / name, target_dir / name
            if _unchanged(source, target):
                continue
            target_dir.mkdir(parents=True, exist_ok=True)
            shutil.copy2(source, target)
            copied += 1
    if not quiet:
        say(f"synced {src} -> {dst} ({copied} files changed)")
    return copied


def make_zip(src_dir, zip_path, store=(".pt", ".pth")):
    """Zip the contents of src_dir (paths relative to it) to zip_path, atomically: the previous zip stays valid until the
    new one is complete. Checkpoints are stored uncompressed (they do not compress, and it saves CPU)."""
    src_dir, zip_path = Path(src_dir), Path(zip_path)
    tmp = zip_path.with_name(zip_path.name + ".tmp")
    with zipfile.ZipFile(tmp, "w", zipfile.ZIP_DEFLATED) as archive:
        for dirpath, dirnames, filenames in os.walk(src_dir):
            dirnames.sort()
            for name in sorted(filenames):
                path = Path(dirpath) / name
                kind = zipfile.ZIP_STORED if path.suffix in store else zipfile.ZIP_DEFLATED
                archive.write(path, path.relative_to(src_dir).as_posix(), compress_type=kind)
    os.replace(tmp, zip_path)
    return zip_path


def prior_gpu_names(runs_dir, exp):
    """{gpu name: [splits]} of the finished splits already on disk for this experiment ID.

    Matches the run folder named like the experiment (a group such as 'E2' matches 'E2-*'), and folders whose config.json
    says they were requested as `exp` (the aliases: E1-U is the E0-rect folder, E1-WW is the E1-WF folder)."""
    runs_dir, found = Path(runs_dir), {}
    if not runs_dir.is_dir():
        return found
    for exp_dir in sorted(p for p in runs_dir.iterdir() if p.is_dir() and p.name not in ("sweep", "scratch")):
        by_name = exp_dir.name == exp or exp_dir.name.startswith(exp + "-")
        for split_dir in sorted(exp_dir.glob("split_*")):
            if not (split_dir / "metrics.json").is_file() or not (split_dir / "env.json").is_file():
                continue
            if not by_name:
                try:
                    config = json.loads((split_dir / "config.json").read_text())
                except (OSError, ValueError):
                    continue
                if exp not in (config.get("requested_exp_id"), (config.get("experiment") or {}).get("exp_id")):
                    continue
            try:
                gpu = json.loads((split_dir / "env.json").read_text()).get("gpu")
            except (OSError, ValueError):
                continue
            if gpu:
                found.setdefault(gpu, []).append(split_dir.name.replace("split_", ""))
    return found


def hardware_warnings(exp, prior, current_names):
    """Messages when finished splits of `exp` ran on a different GPU model than the one about to be used."""
    messages = []
    current = sorted(set(current_names))
    other = {gpu: splits for gpu, splits in prior.items() if gpu not in current}
    for gpu, splits in other.items():
        messages.append(
            f"{exp}: split(s) {', '.join(splits)} already ran on {gpu!r}, but this session uses {', '.join(map(repr, current))}. "
            "All 5 splits of one experiment should run on the same GPU type, otherwise the mean +- std mixes hardware "
            "(different speed and, in rare cases, slightly different numbers). Consider rerunning those splits with --force "
            "or noting the mix in the report.")
    if len(prior) > 1:
        messages.append(f"{exp}: its finished splits already come from different GPU models: {sorted(prior)}.")
    return messages


# ---- batch plans (one function per name in BATCHES_TO_RUN) ---------------------------------------------------

SWEEP_EXPERIMENT = {"convnext_tiny": "E2-cnx-U", "swin_t": "E2-swin-U"}  # the E2 sweeps (the CLI's SWEEP_EXPERIMENT)


def plan_batch1(runner):
    """Batch 1 (done, results committed): ResNet50 sweep and the three E0 rows. Kept for reproducibility."""
    runner.run_sweep("E0-rect")
    runner.choose_lr("resnet50")
    for exp in ("E0-squash", "E0-pad", "E0-rect"):
        runner.run_batch(exp)


def plan_batch2(runner):
    """Batch 2: E1 rows. E1-U and E1-WW are aliases and train nothing (five 'skipped' lines each)."""
    runner.show_choices("input_mode", "lr.resnet50")
    runner.run_batch("E1-U")
    runner.run_parallel(["E1-WF", "E1-FF"])
    runner.run_batch("E1-WW")


def plan_batch3_sweeps(runner):
    """The ConvNeXt-T and Swin-T learning-rate sweeps (split 0, lr 1e-4 and 3e-4) on two GPUs, then choose-lr for each.

    Backbones whose learning rate is already in choices.json are skipped. choose-lr runs after both sweeps finished, one
    after the other, so two processes never write choices.json at the same time."""
    todo = [(backbone, exp) for backbone, exp in SWEEP_EXPERIMENT.items() if not runner.has_choice(f"lr.{backbone}")]
    if not todo:
        say("batch3-sweeps: lr.convnext_tiny and lr.swin_t are already in choices.json - nothing to sweep")
        return
    runner.run_sweeps_parallel([exp for _, exp in todo])
    for backbone, _ in todo:
        runner.choose_lr(backbone)


def plan_batch3_core(runner):
    """Batch 3 core: sweeps (if needed), then E2-cnx-U and E2-swin-U on all 5 splits, one experiment per GPU."""
    plan_batch3_sweeps(runner)
    runner.show_choices("input_mode", "lr.resnet50", "lr.convnext_tiny", "lr.swin_t")
    runner.run_parallel(["E2-cnx-U", "E2-swin-U"])


def plan_batch3_wf(runner):
    """Batch 3 cross-domain rows (cut first if behind schedule): E2-cnx-WF and E2-swin-WF, one per GPU."""
    runner.show_choices("input_mode", "lr.resnet50", "lr.convnext_tiny", "lr.swin_t")
    runner.run_parallel(["E2-cnx-WF", "E2-swin-WF"])


def plan_batch4(runner):
    """Batch 4, Method A (composites, recipe v2): the split-0 comparison of composites per epoch and choose-composites
    (skipped once composite_n_factor is in choices.json), then E3-A on all 5 splits, one split per GPU at a time (split 0
    is the reused sweep run and is skipped)."""
    runner.show_choices("input_mode", "backbone", "lr.swin_t")
    if runner.has_choice("composite_n_factor"):
        say("batch4: composite_n_factor is already in choices.json - no sweep")
    else:
        runner.run_composite_sweep()
        runner.choose_composites()
    runner.show_choices("input_mode", "backbone", "lr.swin_t", "composite_n_factor")
    runner.run_splits_parallel("E3-A")


def plan_batch4_wf(runner):
    """Batch 4 cross-domain row (optional, Tier 4): E3-A-WF, white + composites, tested on field. Needs composite_n_factor
    (run batch4 first)."""
    runner.show_choices("input_mode", "backbone", "lr.swin_t", "composite_n_factor")
    runner.run_splits_parallel("E3-A-WF")


def plan_sanity(runner):
    """Phase 03 sanity run (done): E0-rect, split 0, lr 3e-4. Scratch output, never a result."""
    runner.script("run", "--exp", "E0-rect", "--splits", "0", "--lr", "3e-4")
    print((runner.cfg.runs_dir / "scratch" / "E0-rect_dev" / "split_0" / "history.csv").read_text())


BATCH_PLANS = {
    "sanity": plan_sanity,
    "batch1": plan_batch1,
    "batch2": plan_batch2,
    "batch3-sweeps": plan_batch3_sweeps,
    "batch3-core": plan_batch3_core,
    "batch3-wf": plan_batch3_wf,
    "batch4": plan_batch4,
    "batch4-wf": plan_batch4_wf,
}
BATCH_ALIASES = {"batch3": ["batch3-core", "batch3-wf"]}


def expand_batches(names):
    """Resolve aliases ('batch3' = core + wf), keeping order and dropping duplicates."""
    expanded = []
    for name in names:
        for item in BATCH_ALIASES.get(name, [name]):
            if item not in expanded:
                expanded.append(item)
    return expanded


def validate_batches(names):
    """Fail before any long step on a typo in BATCHES_TO_RUN."""
    bad = [n for n in names if n not in BATCH_PLANS and n not in BATCH_ALIASES and not n.startswith(("run:", "parallel:"))]
    if bad:
        raise RunnerError(f"unknown entries in BATCHES_TO_RUN: {bad}. Valid: {sorted([*BATCH_PLANS, *BATCH_ALIASES])} and 'run:<EXP>', "
                          "'parallel:<EXP>,<EXP>'.")
    for name in names:
        if name.startswith("parallel:") and len([e for e in name[9:].split(",") if e.strip()]) < 2:
            raise RunnerError(f"{name!r}: parallel: needs at least two experiment IDs separated by commas")
        if name.startswith("run:") and not name[4:].strip():
            raise RunnerError(f"{name!r}: run: needs an experiment ID")


def plan_for(name):
    """The plan function for a BATCHES_TO_RUN entry, including the generic run:/parallel: forms."""
    if name in BATCH_PLANS:
        return BATCH_PLANS[name]
    if name.startswith("run:"):
        return lambda runner: runner.run_batch(name[4:].strip())
    if name.startswith("parallel:"):
        return lambda runner: runner.run_parallel([e.strip() for e in name[9:].split(",") if e.strip()])
    raise RunnerError(f"unknown batch {name!r}")


def run_selected(runner, name, selected):
    """Run batch `name` if it is in `selected`. Outputs are synced (and out.zip rebuilt) even when the batch fails,
    and a failure is re-raised afterwards so that Run All stops visibly."""
    selected = expand_batches(selected)
    if name not in selected:
        say(f"[{name}] is not in BATCHES_TO_RUN {list(selected)}: skipped (add it to the list in the first cell, or call "
            "the plan directly)")
        return False
    say(f"\n===== {name} =====")
    try:
        plan_for(name)(runner)
    finally:
        try:
            runner.sync_runs()
        except Exception as exc:  # noqa: BLE001 - never hide the batch's own error behind a sync error
            say(f"final sync after {name} failed: {exc}")
    return True


def run_generic(runner, selected):
    """Run every 'run:' and 'parallel:' entry of `selected`, in list order (the cell for experiments without their own cell)."""
    ran = False
    selected = expand_batches(selected)
    for name in selected:
        if name.startswith(("run:", "parallel:")):
            ran = run_selected(runner, name, selected) or ran
    if not ran:
        say("no run:/parallel: entries in BATCHES_TO_RUN: nothing to do here")

In [ ]:
#| lib
# Helpers 5/5: the Runner (streaming subprocess, sync, batches, two experiments at once)
_DONE = object()
_EOF = object()
STREAM_DRAIN_SECONDS = 3.0  # after the command exits: time to print what its pipe still holds


class Runner:
    """Runs the generated script and keeps the results safe. One instance per session (`R` in the notebook).

    popen is injectable so that the tests can replace the subprocess; gpus is the list from parse_gpus()."""

    def __init__(self, cfg, gpus=(), popen=subprocess.Popen, python=None, keep_checkpoints=False, workers_parallel=2):
        self.cfg = cfg
        self.gpus = list(gpus)
        self.popen = popen
        self.python = str(sys.executable if python is None else python)
        self.keep_checkpoints = keep_checkpoints
        self.workers_parallel = workers_parallel
        self._sync_lock = threading.Lock()
        self._queue = None  # set while threads run: worker output is printed by the calling thread
        self._owner = None
        self._procs = set()

    # ---- output ----------------------------------------------------------------------------------------------
    def emit(self, text, prefix=""):
        line = prefix + str(text).rstrip("\n")
        if self._queue is not None and threading.get_ident() != self._owner:
            self._queue.put(line)  # Jupyter shows prints of worker threads unreliably: the cell's thread prints them
        else:
            say(line)

    def stream(self, cmd, prefix="", env_extra=None):
        """Run a command (argument list, no shell: dataset paths contain spaces) and print its output live. Raises on failure.

        The command runs in its own process group, and its output is read on a helper thread, so the end of the command
        is noticed when the command itself exits, not when its pipe closes: a worker process it leaves behind (which
        still holds the pipe) cannot hang the cell. Whatever is left of the group afterwards is killed, and on an
        interrupt the whole group is."""
        cmd = [str(part) for part in cmd]
        self.emit("$ " + " ".join(cmd) + (f"   [{', '.join(f'{k}={v}' for k, v in env_extra.items())}]" if env_extra else ""), prefix)
        env = {**os.environ, "PYTHONUNBUFFERED": "1", **(env_extra or {})}
        proc = self.popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, errors="replace", env=env,
                          start_new_session=True)
        self._procs.add(proc)
        lines = queue.Queue()

        def pump():
            try:
                for line in proc.stdout:
                    lines.put(line)
            finally:
                lines.put(_EOF)

        threading.Thread(target=pump, daemon=True).start()
        try:
            code = self._follow(proc, lines, prefix)
        finally:
            kill_process_group(proc)
            self._procs.discard(proc)
        if code != 0:
            raise RunnerError(f"command failed (exit code {code}): " + " ".join(cmd[1:]))

    def _follow(self, proc, lines, prefix):
        """Print the lines of ``proc`` until its pipe closes or, if something else keeps the pipe open, until ``proc``
        has exited and the pipe stayed quiet for STREAM_DRAIN_SECONDS. Returns the exit code."""
        quiet_since = None
        while True:
            try:
                item = lines.get(timeout=0.5)
            except queue.Empty:
                item = None
            if item is _EOF:
                return proc.wait()
            if item is not None:
                self.emit(item, prefix)
                quiet_since = None
                continue
            code = proc.poll()
            if code is None:
                continue
            quiet_since = quiet_since or time.monotonic()
            if time.monotonic() - quiet_since >= STREAM_DRAIN_SECONDS:
                self.emit(f"note: the command exited (code {code}) but left processes holding its output; stopping them", prefix)
                return code

    def script(self, *args, gpu=None, num_workers=None, prefix=""):
        """python group3-starvenus.py --data-root <root> --device <cuda|auto> <args>

        gpu=k pins the process to physical GPU k (CUDA_VISIBLE_DEVICES=k): inside it the GPU is cuda:0, which is what
        `--device cuda` uses, and torch.cuda.get_device_name(0) in env.json is the right card."""
        cmd = [self.python, self.cfg.script_path, "--data-root", self.cfg.data_root, "--device", self.cfg.device, *args]
        if num_workers is not None:
            cmd += ["--num-workers", num_workers]
        env_extra = {} if gpu is None else {"CUDA_VISIBLE_DEVICES": str(gpu), "CUDA_DEVICE_ORDER": "PCI_BUS_ID"}
        self.stream(cmd, prefix, env_extra)

    # ---- persistence -----------------------------------------------------------------------------------------
    def restore_runs(self):
        """Colab: bring finished runs and results (choices.json) back from Drive, so that `run` skips finished splits after
        a disconnect. Kaggle and local: nothing to do (the runs were restored from the code dataset during setup)."""
        if self.cfg.persist != "drive":
            return
        cfg = self.cfg
        if cfg.drive_runs.is_dir():
            sync_tree(cfg.drive_runs, cfg.runs_dir, say=self.emit)
        if (cfg.drive_work / "results").is_dir():
            sync_tree(cfg.drive_work / "results", cfg.work / "results", say=self.emit)

    def sync_choices(self):
        """Copy _work/results/choices.json (small) to the persistent place."""
        cfg = self.cfg
        if not cfg.choices_path.is_file():
            return
        target = (cfg.drive_work if cfg.persist == "drive" else cfg.out_dir) / "results" / "choices.json"
        target.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(cfg.choices_path, target)

    def sync_runs(self, keep_checkpoints=None):
        """Copy _work/runs/ (finished splits and sweeps, not the scratch folder) and choices.json to the persistent place.

        Colab: My Drive/csci935/_work. Kaggle: /kaggle/working/out, then /kaggle/working/out.zip is rebuilt, so a session that
        dies still leaves a complete download. Safe to call from several threads."""
        keep = self.keep_checkpoints if keep_checkpoints is None else keep_checkpoints
        cfg = self.cfg
        with self._sync_lock:
            if cfg.persist == "drive":
                sync_tree(cfg.runs_dir, cfg.drive_runs, keep_checkpoints=keep, skip=("scratch",), say=self.emit)
                self.sync_choices()
            else:
                copied = sync_tree(cfg.runs_dir, cfg.out_dir / "runs", keep_checkpoints=keep, skip=("scratch",), say=self.emit,
                                   quiet=True)
                self.sync_choices()
                self.refresh_zip()
                self.emit(f"synced runs -> {cfg.out_dir / 'runs'} ({copied} files changed), {cfg.out_zip.name} refreshed "
                          f"({cfg.out_zip.stat().st_size / 1e6:.1f} MB)")

    def refresh_zip(self):
        if self.cfg.persist == "outdir":
            self.cfg.out_dir.mkdir(parents=True, exist_ok=True)
            make_zip(self.cfg.out_dir, self.cfg.out_zip)

    # ---- choices.json ----------------------------------------------------------------------------------------
    def _choices(self):
        path = self.cfg.choices_path
        return json.loads(path.read_text()) if path.is_file() else {}

    def has_choice(self, name):
        node = self._choices()
        for part in name.split("."):
            node = node.get(part) if isinstance(node, dict) else None
        return isinstance(node, dict) and "value" in node

    def show_choices(self, *needed):
        """Print choices.json and stop if an entry is missing. Names: 'input_mode', 'backbone', 'lr.resnet50', ..."""
        missing = [name for name in needed if not self.has_choice(name)]
        if missing:
            where = ("My Drive/csci935/_work/results/" if self.cfg.persist == "drive"
                     else "the 'csci935-code' dataset (new version) as choices.json")
            raise RunnerError(f"choices.json has no {', '.join(missing)}. Put the committed "
                              f"Dhan-Shomadhan/_work/results/choices.json from your computer into {where}, then run again.")
        data = self._choices()
        say(str(self.cfg.choices_path))
        for key, entry in data.items():
            values = {k: v["value"] for k, v in entry.items()} if key == "lr" else entry["value"]
            say(f"  {key}: {values}")

    # ---- hardware rule ---------------------------------------------------------------------------------------
    def check_hardware(self, exp, gpus=None):
        """Warn loudly when finished splits of `exp` ran on a different GPU model than the one used now."""
        gpus = self.gpus[:1] if gpus is None else gpus
        names = [g["name"] for g in gpus]
        if not names:
            return []
        messages = hardware_warnings(exp, prior_gpu_names(self.cfg.runs_dir, exp), names)
        for message in messages:
            warn(message)
        return messages

    # ---- batches ---------------------------------------------------------------------------------------------
    def run_batch(self, exp, splits=DEFAULT_SPLITS, keep_checkpoints=None):
        """Train and evaluate one experiment (or group such as E0) split by split, syncing after each split.

        Finished splits are skipped, so running it again after a disconnect resumes. keep_checkpoints=True also keeps
        best.pt (~100-330 MB per split), only needed for Grad-CAM / predict."""
        self.restore_runs()
        self.check_hardware(exp)
        for split in splits:
            self.script("run", "--exp", exp, "--splits", str(split))
            self.sync_runs(keep_checkpoints)

    def run_sweep(self, exp):
        """Split-0 learning-rate sweep (lr 1e-4 and 3e-4) into _work/runs/sweep/. Never counts as a result."""
        self.restore_runs()
        self.script("run", "--exp", exp, "--sweep-lr")
        self.sync_runs()

    def choose_lr(self, backbone):
        """Record the better sweep lr in choices.json (and reuse the winning sweep run as split 0 of the real run folder),
        then persist both."""
        self.script("choose-lr", "--backbone", backbone)
        if self.cfg.persist == "drive":
            sync_tree(self.cfg.work / "results", self.cfg.drive_work / "results", say=self.emit)
        self.sync_runs()  # also pushes the sweep-reused split 0
        say(self.cfg.choices_path.read_text())
        say("-> your local choices.json is the source of truth: copy this one into it and commit")

    def _parallel(self, jobs, keep_checkpoints=None):
        """jobs: [(label, [arg tuples])]. Each job runs its commands one after another on one GPU; with two GPUs two jobs
        run at once, a GPU that finishes early takes the next job. Raises after all jobs ended if any failed."""
        slots = self.gpus[: min(len(self.gpus), len(jobs))]
        pending = queue.Queue()
        for job in jobs:
            pending.put(job)
        self._queue, self._owner = queue.Queue(), threading.get_ident()
        failures = []

        def worker(gpu):
            try:
                while True:
                    try:
                        label, commands = pending.get_nowait()
                    except queue.Empty:
                        break
                    prefix = f"[{label}|gpu{gpu['index']}] "
                    try:
                        for args in commands:
                            self.script(*args, gpu=gpu["index"], num_workers=self.workers_parallel, prefix=prefix)
                            self.sync_runs(keep_checkpoints)
                    except Exception as exc:  # noqa: BLE001 - reported below; the other GPU keeps working
                        failures.append((label, str(exc)))
                        self.emit(f"FAILED: {exc}", prefix)
            finally:
                self._queue.put(_DONE)

        threads = [threading.Thread(target=worker, args=(gpu,), daemon=True) for gpu in slots]
        try:
            for thread in threads:
                thread.start()
            running = len(threads)
            while running:
                item = self._queue.get()
                if item is _DONE:
                    running -= 1
                else:
                    say(item)
        except BaseException:  # Interrupt / cancel: do not leave training processes (or their workers) behind
            for proc in list(self._procs):
                kill_process_group(proc)
            raise
        finally:
            self._queue = None
            for thread in threads:
                thread.join(timeout=30)
        if failures:
            raise RunnerError("failed: " + "; ".join(f"{label}: {message}" for label, message in failures))

    def run_parallel(self, exps, splits=DEFAULT_SPLITS, keep_checkpoints=None):
        """Run experiments at the same time, one per GPU (CUDA_VISIBLE_DEVICES), each with --num-workers
        NUM_WORKERS_PARALLEL. With one GPU, or one experiment, it falls back to run_batch one after the other."""
        exps = list(exps)
        if len(self.gpus) < 2 or len(exps) < 2:
            if len(exps) >= 2:
                warn(f"only {len(self.gpus)} GPU(s) visible: running {exps} one after the other. Select 'GPU T4 x2' to run them at once.")
            for exp in exps:
                self.run_batch(exp, splits, keep_checkpoints)
            return
        self.restore_runs()
        for exp in exps:
            self.check_hardware(exp, self.gpus[:2])
        say(f"running {exps} at once on {len(self.gpus[:len(exps)])} GPU(s), {self.workers_parallel} data workers each")
        self._parallel([(exp, [("run", "--exp", exp, "--splits", str(split)) for split in splits]) for exp in exps], keep_checkpoints)

    def run_sweeps_parallel(self, exps):
        """The split-0 sweeps of several experiments at once, one per GPU (sequential with one GPU)."""
        exps = list(exps)
        if len(self.gpus) < 2 or len(exps) < 2:
            for exp in exps:
                self.run_sweep(exp)
            return
        self.restore_runs()
        say(f"sweeping {exps} at once on {len(self.gpus[:len(exps)])} GPU(s)")
        self._parallel([(exp, [("run", "--exp", exp, "--sweep-lr")]) for exp in exps])

    def run_splits_parallel(self, exp, splits=DEFAULT_SPLITS, keep_checkpoints=None):
        """The splits of one experiment spread over the GPUs, one split per GPU at a time (a GPU that finishes takes the
        next split). Finished splits are skipped, as in run_batch. With one GPU it is run_batch."""
        if len(self.gpus) < 2:
            self.run_batch(exp, splits, keep_checkpoints)
            return
        self.restore_runs()
        self.check_hardware(exp, self.gpus[:2])
        say(f"running {exp} splits {list(splits)} on {len(self.gpus[:2])} GPUs, one split per GPU at a time")
        self._parallel([(f"{exp} s{split}", [("run", "--exp", exp, "--splits", str(split))]) for split in splits], keep_checkpoints)

    def run_composite_sweep(self):
        """Method A's split-0 comparison of composites per epoch (0.5x and 1x the white training count), one run after the
        other on one GPU, into _work/runs/sweep/. Never counts as a result."""
        self.restore_runs()
        self.script("run", "--exp", "E3-A", "--sweep-composites")
        self.sync_runs()

    def choose_composites(self):
        """Record the better factor in choices.json (and reuse the winning sweep run as split 0 of E3-A), then persist both."""
        self.script("choose-composites")
        if self.cfg.persist == "drive":
            sync_tree(self.cfg.work / "results", self.cfg.drive_work / "results", say=self.emit)
        self.sync_runs()
        say(self.cfg.choices_path.read_text())
        say("-> your local choices.json is the source of truth: copy this one into it and commit")

    def finish(self):
        """Final sync (and out.zip). Returns a short listing of what the outputs hold."""
        self.sync_runs()
        if self.cfg.persist == "outdir":
            return summarize_outputs(self.cfg)
        return ""


def summarize_outputs(cfg):
    lines = [f"{cfg.out_zip} ({cfg.out_zip.stat().st_size / 1e6:.1f} MB) holds:"]
    with zipfile.ZipFile(cfg.out_zip) as archive:
        names = archive.namelist()
    lines.append(f"  {len(names)} files; top level: {sorted({n.split('/')[0] for n in names})}")
    for exp, splits in finished_splits(cfg.out_dir / "runs").items():
        lines.append(f"  runs/{exp}: splits {splits}")
    return "\n".join(lines)

In [ ]:
# Setup 1: paths of this platform, Drive mount (Colab), internet (Kaggle), GPU check
validate_batches(BATCHES_TO_RUN)  # a typo fails now, not after an hour of setup
check_python()
CFG = resolve_config(PLATFORM, GROUP_NAME)
set_log_path(CFG.log_path)
if PLATFORM == "colab":
    from google.colab import drive

    drive.mount("/content/drive")

GPUS = query_gpus()
if CFG.require_gpu:
    GPU_NAMES = check_gpus(GPUS, PLATFORM)  # fails early: no GPU, or a P100 (Kaggle: select GPU T4 x2)
else:
    GPU_NAMES = [gpu["name"] for gpu in GPUS]
    say("local mode: no GPU required")
if PLATFORM == "kaggle":
    ensure_internet()  # pip and download-weights need it: Session options > Internet > On (phone-verified account)

R = Runner(CFG, GPUS, keep_checkpoints=KEEP_CHECKPOINTS, workers_parallel=NUM_WORKERS_PARALLEL)
SCRIPT = CFG.script_name
DEVICE = CFG.device
DATA_ROOT, WORK = CFG.data_root, CFG.work
# Short names for the batch cells and for interactive use
script, run_batch, run_parallel, run_sweep, run_sweeps_parallel = R.script, R.run_batch, R.run_parallel, R.run_sweep, R.run_sweeps_parallel
choose_lr, show_choices, sync_runs = R.choose_lr, R.show_choices, R.sync_runs


def batch(name):
    """Run the batch `name` if it is in BATCHES_TO_RUN; sync and zip the outputs afterwards (also when it fails)."""
    return run_selected(R, name, BATCHES_TO_RUN)


say(f"{PLATFORM}: data root {DATA_ROOT}, script {CFG.script_path}, results -> "
    f"{CFG.drive_runs if CFG.persist == 'drive' else CFG.out_dir}")

In [ ]:
# Setup 2: the script and requirements.txt
#   Colab: from My Drive/csci935/    Kaggle: from the csci935-code dataset (found by globbing /kaggle/input)    local: the checkout
INPUTS = {}
if PLATFORM == "colab":
    for name in (SCRIPT, "requirements.txt"):
        source = CFG.drive / name
        assert source.exists(), f"missing {source}: upload it to My Drive/csci935/"
        shutil.copy2(source, CFG.script_path.parent / name)
    say(f"copied {SCRIPT} and requirements.txt from {CFG.drive}")
elif PLATFORM == "kaggle":
    INPUTS = discover_kaggle_inputs(CFG.input_root, SCRIPT, EXPECTED_IMAGES)
    CFG.script_path.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(INPUTS["script"], CFG.script_path)
    shutil.copy2(INPUTS["requirements"], CFG.requirements_path)
    say("kaggle inputs:")
    for key in ("code_dir", "data_kind", "data_source", "runs_zip", "runs_dir", "choices"):
        say(f"  {key}: {INPUTS[key]}")
else:
    assert CFG.script_path.is_file(), f"missing {CFG.script_path}; run `python tools/export_py.py` first"

In [ ]:
# Setup 3: pinned packages
#   torch/torchvision: kept when the preinstalled build has the pinned version and CUDA works on every GPU; otherwise the
#   pin is installed (CUDA 13.0 build if the driver is 580 or newer, else CUDA 12.8, else 12.6), with a loud warning if the
#   pin cannot be honoured. Then requirements.txt, with exactly one OpenCV distribution (4.12).
TORCH_RESULT = None
if PLATFORM == "local":
    say("local mode: nothing is installed")
    PROBE = probe_environment()
else:
    PINS = parse_requirements(CFG.requirements_path.read_text())
    TORCH_RESULT = ensure_torch(R.stream, PINS, GPUS[0]["driver"], len(GPUS))
    PROBE = install_requirements(R.stream, CFG.requirements_path, TORCH_RESULT["honoured"], CFG.scratch)
    freeze_path = (CFG.drive / "pip_freeze.txt") if PLATFORM == "colab" else (CFG.out_dir / "pip_freeze.txt")
    if PLATFORM == "kaggle" or not freeze_path.exists():  # Colab: only the first time (use it to pin requirements.txt)
        freeze_path.parent.mkdir(parents=True, exist_ok=True)
        freeze_path.write_text(pip_freeze())
        say(f"recorded {freeze_path}")

In [ ]:
# Setup 4: the dataset (complete copy of the original structure, writable, outside the saved output)
#   Colab: unzip Dhan-Shomadhan.zip from Drive.  Kaggle: unzip the zip, or copy the tree Kaggle extracted (folder names may
#   have been altered, the code never depends on them; what Kaggle did is printed with repr() and recorded in env.json).
if PLATFORM == "colab":
    zip_path = CFG.drive / "Dhan-Shomadhan.zip"
    assert zip_path.exists(), f"missing {zip_path}: upload the original Dhan-Shomadhan.zip to My Drive/csci935/"
    LAYOUT = materialise_dataset("zip", zip_path, DATA_ROOT, CFG.scratch, EXPECTED_IMAGES)
elif PLATFORM == "kaggle":
    LAYOUT = materialise_dataset(INPUTS["data_kind"], INPUTS["data_source"], DATA_ROOT, CFG.scratch, EXPECTED_IMAGES)
    LAYOUT["inputs"] = {key: str(INPUTS[key]) for key in ("code_dir", "data_kind", "data_source", "runs_zip", "runs_dir", "choices")}
else:
    n_images = count_images(DATA_ROOT)
    assert n_images == EXPECTED_IMAGES, f"expected {EXPECTED_IMAGES} images under {DATA_ROOT}, found {n_images}"
    LAYOUT = {"kind": "local", "data_root_top_level": top_level_names(DATA_ROOT)}
    say(f"dataset OK: {n_images} images under {DATA_ROOT}")

In [ ]:
# Setup 5: restore earlier runs and choices.json, so that finished splits are skipped
#   Colab: from Drive.  Kaggle: runs.zip and choices.json from the csci935-code dataset.  local: nothing.
if PLATFORM == "colab":
    R.restore_runs()
elif PLATFORM == "kaggle":
    restore_from_code_dataset(INPUTS, WORK)
say("finished splits found: " + (str(finished_splits(CFG.runs_dir)) or "none"))

In [ ]:
# Setup 6: record the environment next to the runs (GPU, driver, Python, torch, torchvision, OpenCV, whether the pin held)
ENV = build_env_record(CFG, GPUS, PROBE, TORCH_RESULT, LAYOUT)
CFG.env_json_path.parent.mkdir(parents=True, exist_ok=True)
CFG.env_json_path.write_text(json.dumps(ENV, indent=2) + "\n")
say(json.dumps(ENV, indent=2))
if not ENV["python"].startswith("3.12"):
    say(f"note: the project targets Python 3.12; this session runs {ENV['python']} (README P7: package versions, not the interpreter, decide the numbers)")
R.refresh_zip()  # Kaggle/local: out.zip exists from now on, even if the session dies in the first minutes

## 2. Smoke test (must pass before continuing)

In [ ]:
# Smoke test on the original dataset structure: must print the GPU name(s) and PASSED
say(f"GPU: {GPU_NAMES}")
if "smoke" in SKIP_STEPS:
    say("smoke skipped (RUNNER_SKIP)")
else:
    script("smoke")

## 3. Caches: `download-weights` and `prepare` (both idempotent)

Colab also restores `cache/`, `weights/`, `masks/`, `results/` from Drive and saves new ones back. Kaggle rebuilds them every session (a few minutes) because they are not part of the saved output.

In [ ]:
# Caches: Colab restores them from Drive if present and saves new ones; Kaggle and local rebuild what is missing
PERSISTED = ("cache", "weights", "masks", "results")  # Colab: restored from / saved to My Drive/csci935/_work between sessions
if PLATFORM == "colab":
    for name in PERSISTED:
        if (CFG.drive_work / name).is_dir():
            sync_tree(CFG.drive_work / name, WORK / name)

if "prepare" in SKIP_STEPS:
    say("download-weights and prepare skipped (RUNNER_SKIP)")
else:
    script("download-weights")  # idempotent: prints the url and stored path of every weight file
    script("prepare")  # idempotent: skips cache and masks that already exist and are valid

if PLATFORM == "colab":
    for name in PERSISTED:
        if (WORK / name).is_dir() and not (CFG.drive_work / name).is_dir():
            sync_tree(WORK / name, CFG.drive_work / name)

## 4. PHASE 03 SANITY RUN (done; name `sanity`)

One real run of `E0-rect` on split 0 with the full recipe at lr 3e-4. Output goes to `_work/runs/scratch/E0-rect_dev/` and is
never a result. Only runs if `"sanity"` is in `BATCHES_TO_RUN`.

In [ ]:
batch("sanity")

## 4b. BATCH 1 - E0 input-size ablation (ResNet50, unified)  [DONE, results committed]

Name `batch1`. Nothing to run any more; the cell stays for reproducibility (finished splits are skipped). It sweeps the
ResNet50 learning rate, runs `choose-lr`, then `E0-squash`, `E0-pad`, `E0-rect`. Gate G0 (`gate g0`, local) chose `rect`.

In [ ]:
batch("batch1")

## 4c. BATCH 2 - E1 (ResNet50 baseline and domain experiments)  [name `batch2`; planned for the local M2]

Needs `input_mode` and `lr.resnet50` in `choices.json` (the plan stops with a message otherwise). Order: `E1-U` (alias of the
winning E0 row, trains nothing), `E1-WF` and `E1-FF` (two at once on T4 x2), `E1-WW` (alias of `E1-WF`, trains nothing).
Estimated on one T4 (measured 19.7 s/epoch for ResNet50 rect, about 28 epochs per run): E1-WF about 32 min, E1-FF about 16 min.

In [ ]:
batch("batch2")

## 4d. BATCH 3 - E2 (ConvNeXt-T and Swin-T)  [DONE, results committed]

| Name | What |
|---|---|
| `batch3-sweeps` | learning-rate sweeps (split 0, lr 1e-4 and 3e-4) for ConvNeXt-T and Swin-T, one per GPU, then `choose-lr` for each (sequentially, after both sweeps) |
| `batch3-core` | the sweeps (skipped for a backbone whose lr is already in `choices.json`), then `E2-cnx-U` and `E2-swin-U` on all 5 splits, one experiment per GPU |
| `batch3-wf` | `E2-cnx-WF` and `E2-swin-WF` (first to cut if behind schedule, README cut lines), one per GPU |
| `batch3` | `batch3-core` then `batch3-wf` |

`choose-lr` also copies the winning sweep run to split 0 of the real run folder (`E2-cnx-U`, `E2-swin-U`) when it is the same
training (identical config fingerprint and split), so the real runs train only splits 1 to 4. Each experiment syncs after every
split; if the session dies, run the notebook again and finished splits are skipped.

Estimated time (measured T4 ResNet50: 19.7 s/epoch, about 28 epochs per run; ConvNeXt-T and Swin-T are not measured, assumed 2x):
one T4 needs about 3.3 h for `batch3-core` and about 5.5 h with the WF rows; with two GPUs in parallel about half of that.
Check the first epoch time of the sweep and rescale. If a model runs out of memory at batch 32 it retries by itself with batch 16 x 2 accumulation.

In [ ]:
batch("batch3-sweeps")

In [ ]:
batch("batch3-core")

In [ ]:
batch("batch3-wf")

## 4e. BATCH 4 - Method A (E3-A, composites)

| Name | What |
|---|---|
| `batch4` | the split-0 comparison of composites per epoch (0.5x and 1x the white training count, one run after the other on one GPU), `choose-composites`, then `E3-A` on all 5 splits, one split per GPU at a time |
| `batch4-wf` | optional (Tier 4): `E3-A-WF`, white + composites, tested on field ("does synthesis close the W->F gap?"); needs `batch4` first |

Needs `input_mode`, `backbone` (gate G1: `swin_t`) and `lr.swin_t` in `choices.json`: upload the committed one with the code.
The composites use recipe v2 and the reviewed reject list, both inside `group3-starvenus.py`. `choose-composites` copies the
winning sweep run to split 0 of `E3-A`, so the real run trains splits 1 to 4. Estimated on T4 x2 (Swin-T measured about
5-6 min per split without composites, assumed 1.7x with them): sweep about 20 min, then about 20 min for splits 1-4.

In [ ]:
batch("batch4")

In [ ]:
batch("batch4-wf")

# 5. OTHER EXPERIMENTS BY NAME

Any experiment ID of the matrix in `docs/instructions/README.md` can be run by name from `BATCHES_TO_RUN`:

- `"run:X-domain"` runs that experiment on all 5 splits, one split at a time, synced after each.
- `"parallel:E3-A,E3-A-WF"` runs two experiments at once, one per GPU (T4 x2).

The cell below runs every `run:` and `parallel:` entry, in list order. Run a cell again after a disconnect: finished splits are skipped.

In [ ]:
run_generic(R, BATCHES_TO_RUN)

### Interactive use

Any helper can be called directly in a new cell, whatever `BATCHES_TO_RUN` says:

```python
run_batch("E1-FF")                           # one experiment, 5 splits, synced after each split
run_parallel(["E2-cnx-WF", "E2-swin-WF"])    # two experiments at once, one per GPU (T4 x2)
run_batch("E3-A", keep_checkpoints=True)     # keep best.pt (needed for Grad-CAM / predict)
show_choices("input_mode", "lr.resnet50")    # stop with a message if choices.json lacks an entry
```

## 6. Finish

In [ ]:
# Final sync. Kaggle/local: rebuild out.zip and list what it holds. Colab: also push the small folders to Drive.
print(R.finish())
if PLATFORM == "colab":
    for name in ("results", "figures", "splits"):
        if (WORK / name).is_dir():
            sync_tree(WORK / name, CFG.drive_work / name, keep_checkpoints=False)
elif PLATFORM == "kaggle":
    say(f"download: Output tab > {CFG.out_zip.name}  (or the out/ folder)")